# Look Eyrie snow depth

Hourly SNOWPACK outputs from the two lapse-adjusted forcing runs are compared with NRCS SNOTEL 768 snow depth. Both simulations use the same soil profile, snow-free initialization, and 1.2 °C phase threshold. Statistics use the same hours with valid observations and output from both runs. Positive bias indicates deeper modeled snow. Both runs use neutral atmospheric stability after the Holtslag scheme failed to converge during light winds in the mixed run.

The station series retains the prepared NRCS values. The export reports 5.79 m on multiple hours in July 2023. Check the station record before using the error metrics to infer model performance.

Both simulations use a 60-minute integration step from the start through the 2023-10-13 18:00 UTC checkpoint, then a 30-minute step with hourly output. The shorter step resolved the HRRR-AK soil-temperature solver failure.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
default_site = repo.parents[1] / "models" / "run_snowpack" / "sites" / "lookeyrie"
site = Path(os.environ.get("LOOK_EYRIE_SNOWPACK_SITE", default_site)).expanduser()
input_dir = site / "input"
if not input_dir.is_dir():
    raise FileNotFoundError(f"Look Eyrie inputs are missing at {input_dir}. Set LOOK_EYRIE_SNOWPACK_SITE.")
runs = {
    "HRRR-AK": "hrrrak_lapserate_lookeyrie",
    "Met mixed": "met_hrrrak_lapserate_lookeyrie",
}
colors = {"Station": "#000000", "HRRR-AK": "#1764aa", "Met mixed": "#238343"}

def read_smet(path, columns=None):
    header = {}
    with path.open() as stream:
        for line in stream:
            if line.strip() == "[DATA]":
                break
            if "=" in line:
                key, value = line.split("=", 1)
                header[key.strip()] = value.strip()
        fields = header["fields"].split()
        data = pd.read_csv(stream, sep=r"\s+", names=fields,
                           usecols=None if columns is None else ["timestamp", *columns],
                           na_values=[float(header["nodata"])])
    data.index = pd.DatetimeIndex(pd.to_datetime(data.pop("timestamp"), utc=True), name="UTC")
    return data, header


In [ ]:
observed = pd.read_csv(input_dir / "look_eyrie_snow_depth.csv",
                       index_col="timestamp", parse_dates=True).snow_depth_m.rename("Station")
checkpoint = pd.Timestamp("2023-10-13T18:00:00Z")
simulations = {}
for label, run in runs.items():
    segments = {}
    for period in ("hourly", "half_hour"):
        output = site / "output" / period / f"{run}_soil.smet"
        model, header = read_smet(output, ["HS_mod"])
        field = header["fields"].split().index("HS_mod")
        scale = float(header["units_multiplier"].split()[field])
        segments[period] = model.HS_mod * scale
    hourly = segments["hourly"].loc[:checkpoint]
    restarted = segments["half_hour"]
    if hourly.index[-1] != checkpoint or restarted.index[0] != checkpoint:
        raise ValueError(f"{label} output does not meet at the saved checkpoint")
    simulation = pd.concat([hourly, restarted.loc[restarted.index > checkpoint]])
    if not simulation.index.equals(observed.index) or not simulation.notna().all():
        raise ValueError(f"{label} output does not cover the complete hourly evaluation period")
    simulations[label] = simulation

depth = pd.concat([observed, pd.DataFrame(simulations)], axis=1)
paired = depth.dropna()
assert not paired.empty
error = paired[list(runs)].sub(paired["Station"], axis=0)
display(pd.DataFrame({"Paired hours": len(paired), "Bias (m)": error.mean(),
                      "MAE (m)": error.abs().mean(), "RMSE (m)": error.pow(2).mean().pow(0.5)}).round(3))
print(f"Paired observations from {paired.index[0]} to {paired.index[-1]}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4), layout="constrained")
for label in depth.columns:
    ax.plot(depth.index, depth[label], color=colors[label], lw=0.9, label=label,
            zorder=3 if label == "Station" else 2)
ax.set(xlabel="UTC date", ylabel="Snow depth (m)", ylim=(0, None))
ax.legend()
ax.grid(alpha=0.2)
plt.show()

Water years start on October 1. The first and last water years are partial. Peaks use only hours with observations and both simulations present.

In [ ]:
water_year = paired.index.year + (paired.index.month >= 10).astype(int)
seasonal = paired.groupby(water_year).max()
seasonal.index.name = "Water year"
display(seasonal.round(2))
ax = seasonal.plot.bar(color=colors, figsize=(10, 4), rot=0)
ax.set(ylabel="Peak snow depth on paired hours (m)", xlabel="Water year")
plt.tight_layout()
plt.show()